### Loss Function(손실함수)
: 모델의 예측값이 실제 정답과 얼마나 다른지 수치로 계산하는 함수
  모델의 목표는 손실값을 줄이는 것

- 기본옹어
    - Prediciton : 모델이 예측한 값
    - Target : 정답
    - Loss : 예측값과 정답의 차이를 계산한 값

- 손실함수의 역할
    - 예측값과 정답의 차이를 수치로 표현
    - 모델이 얼마나 잘 예측했는지 평가할 수 있음
    - 자동 미분을 통해 weight와 bias를 업데이트할 때 사용할 기울기를 계산할 수 있음

#### 1. MSELoss(Mean Squared Error)
: 평균 제곱 오차를 계산하는 손실함수
예측값과 정답의 차이를 제곱한 뒤 평균을 계산함

- 특징
    - 오차를 제곱하므로 큰 오차에 더 큰 패널티를 부여함
    - 손실값이 0 이상임
    - 예측값이 정답과 같으면 손실값은 0
    - 주로 회귀문제에서 사용

- 사용방법
loss_fn = mm.MSELoss()
loss = loss_fn(prediciton, target)

기본설정 reduction = 'mean'은 모든 원소의 제곱오차의 평균


In [2]:
import torch
import torch.nn as nn

In [4]:
prediction = torch.tensor([2., 4, 6])
target = torch.tensor([1., 5, 4])

loss_fn = nn.MSELoss()
loss = loss_fn(prediction, target)

print(prediction)
print(target)
print(loss)

loss2 = ((target - prediction)**2).mean()
print('직접 계산한 MSE : ', loss2)

tensor([2., 4., 6.])
tensor([1., 5., 4.])
tensor(2.)
직접 계산한 MSE :  tensor(2.)


#### - reduction 옵션
|옵션|의미|
| --- | --- |
| mean|모든 원소의 손실 평균(기본값)|
|sum| 모든 원소의 손실 합|
|none|원소별 손실을 그대로 반환|

** none은 데이터별 손실을 따로 확인하거나 가중치를 적용할 때 유용함

In [32]:
prediction = torch.tensor([2., 4, 6])
target = torch.tensor([1., 5, 4])

loss_mean = nn.MSELoss(reduction='mean')(prediction, target)
loss_sum = nn.MSELoss(reduction='sum')(prediction, target)
loss_none = nn.MSELoss(reduction='none')(prediction, target)

print('mean :', loss_mean)
print('sum :', loss_sum)
print('none :', loss_none)

mean : tensor(2.)
sum : tensor(6.)
none : tensor([1., 1., 4.])


#### 2. L1Loss(Mean Absolute Error)
: 예측값과 정답의 차이를 절댓값으로 적용한 뒤 평균 계산

- 특징
    - 오차에 절댓값 사용
    - 손실값 0 이상
    - 예측값과 정답이 같으면 손실값은 0
    - MSE보다 큰 오차에 대한 패널티가 상대적으로 작음
    - 주로 회귀문제에서 사용

- 사용법
loss_fn = nn.L1Loss()
loss = loss_fn(prediction, targe)

In [7]:
prediction = torch.tensor([2., 4, 6])
target = torch.tensor([1., 5, 4])

loss_fn = nn.L1Loss()
loss = loss_fn(prediction, target)

print('예측값 : ', prediction)
print('정답 :', target)
print('L1 Loss :', loss)

loss2 = torch.abs(target - prediction).mean()
print('직접 계산한 MAE : ', loss2)

예측값 :  tensor([2., 4., 6.])
정답 : tensor([1., 5., 4.])
L1 Loss : tensor(1.3333)
직접 계산한 MAE :  tensor(1.3333)


- 큰 오차를 더 강하게 반영하고 싶다면 MSE를 , 큰 오차나 이상치의 영향을 상대적으로 적게받고싶으면 L1을 고려

In [9]:
prediction = torch.tensor([10., 10, 10])
target = torch.tensor([9., 8, 0])

mse_loss = nn.MSELoss()(prediction, target)
l1_loss = nn.L1Loss()(prediction, target)

print('MSE Loss :' , mse_loss)
print('L1 Loss :', l1_loss)

MSE Loss : tensor(35.)
L1 Loss : tensor(4.3333)


#### 3. BCELoss(Binary Cross Entropy)
: 이진분류에서 사용하는 손실 함수

**중요**
모델이 Sigmoid를 통과한 확률값을 입력으로 받아야 함

nn.BCELoss()(prediciton, target)<br>
 &emsp;- prebability : 0~1 사이의 확률값<br>
 &emsp;- target : 정답. 일반적으로 0 또는 1이며 실수형 텐서로 전달함<br>

 - 실제 정답이 1인데 확률이 1에 가까우면 손실이 작아지고, 0에 가까우면 손실 커짐.
 - 극단적인 값에서 NaN 발생 위험 있음

-

In [11]:
probability = torch.tensor([0.9, 0.2, 0.7, 0.1])
target = torch.tensor([1., 0, 1, 0])

loss = nn.BCELoss()(probability, target)

print('예측확률 :', probability)
print('정답 : ', target)
print('BCE Loss :', loss)


예측확률 : tensor([0.9000, 0.2000, 0.7000, 0.1000])
정답 :  tensor([1., 0., 1., 0.])
BCE Loss : tensor(0.1976)


In [12]:
import math
print(math.log(0.9))

-0.10536051565782628


#### 4. BCEWithLogitsLoss
: Sigmoid와 BCELoss 계산을 하나로 결합한 손실함수

- 중요한 차이
    - BCELoss() : Sigmoid를 적용한 확률값을 입력
    - BCEWithLogitsLoss() : Sigmoid를 적용하지 않은 Logits을 입력으로 받음

- 사용법
    nn.BECWithLogitsLoss()(logits, target)

- 왜 사용하나
    Sigmoid와 BCELoss계산을 결합하여 수치적으로 더 안정적인 방식으로 손실을 계산할 수 있음<br>
    내부적으로 안정적인 연산 수행(극단적인 값에서 NaN발생 위험이 적음)

In [14]:
logits = torch.tensor([2, -1.5, 1, -2])
target = torch.tensor([1., 0, 1, 0])

loss = nn.BCEWithLogitsLoss()(logits, target)

print(logits)
print(loss)

tensor([ 2.0000, -1.5000,  1.0000, -2.0000])
tensor(0.1921)


In [ ]:
logits = torch.tensor([1.2, -0.7, 2, -1.5])
target = torch.tensor([1., 0, 1, 0])

#BCELoss
prob = torch.sigmoid(logits)
loss_b = nn.BCELoss()(prob, target)

# BCEWithLogitsLoss
loss_l = nn.BCEWithLogitsLoss()(logits, target)

print(loss_b)
print(loss_l)
print('두 손실값이 같은가? : ', torch.allclose(loss_b, loss_l))


tensor(0.2487)
tensor(0.2487)
두 손실값이 같은가? :  True


In [24]:
logits_a = torch.tensor([1000.], requires_grad=True)
target = torch.tensor([0.])

probability = torch.sigmoid(logits_a)
loss_a = nn.BCELoss()(probability, target)

loss_a.backward()

logits_b = torch.tensor([1000.])
loss_b = nn.BCEWithLogitsLoss()(logits_b, target)

print(loss_a)
print(loss_b)

tensor(100., grad_fn=<BinaryCrossEntropyBackward0>)
tensor(1000.)


In [26]:
#데이터 3개, 클래스 4개에 대한 모델이 출력이 되엇음(logits)
logits = torch.tensor([
    [2,1,0.1,0.5],
    [0.2,2.5,1,0.3],
    [0.1,0.4,0.8,2]
])

# 각 데이터의 정답 클래스 인덱스
target = torch.tensor([0, 1, 3], dtype = torch.long)

loss = nn.CrossEntropyLoss()(logits, target)
print(loss)

tensor(0.4724)


#### 문제 유형에 따른 손실함수 선택
|문제 유형|대표 손실함수|모델 선택|
| --- | --- | --- |
|회귀|MSELoss, L1Loss|일반적인 실수값|
|이진분류|BCELoss|Sigmoid 적용 확률|
|이진분류|BCEWithLogitsLoss|Sigmoid 적용 전 Logits|
|다중분류|CrossEntropyLoss|Sigmoid 적용 전 Logits|

In [31]:
# 각 손실함수의 입력 형태에 따른 확인

# 회귀
regression_prediction = torch.tensor([2.5, 3])
regression_torch = torch.tensor([3, 2.])

print('MSE :', nn.MSELoss()(regression_prediction, regression_torch).item())
print('L1 :', nn.L1Loss()(regression_prediction, regression_torch).item())

# 이진분류
binary_logits = torch.tensor([1., -1])
binary_target = torch.tensor([1, 0.])

print('BCEWithLogits : ', nn.BCEWithLogitsLoss()(binary_logits, binary_target).item())

# 다중분류
multi_logits = torch.tensor([[1, 2, 0.5],[2, 0.5, 1]])
multi_target = torch.tensor([1, 0], dtype=torch.long)

print('다중 분류 :')
print('CrossEntropy :', nn.CrossEntropyLoss()(multi_logits, multi_target).item())

MSE : 0.625
L1 : 0.75
BCEWithLogits :  0.3132617175579071
다중 분류 :
CrossEntropy : 0.4643687903881073
